# Phase 4 — Open-Set Recognition
19. Distance-to-nearest-prototype rejection rule
20. Threshold determined on validation set (not a fixed guess)
21. Evaluate: Known-F1, Unknown-F1, AUROC -- across shot settings and image-only vs image+metadata

**Open-set design decision (confirmed):** `Stage 4 ROP` and `Stage 5 ROP` are held out as the "unknown" classes.
- They are excluded entirely from prototype construction (support set) and from classifier training.
- At test time, the model only sees them as "an image that should be rejected as unknown."
- Remaining 6 classes (A-ROP, Laser-treated ROP, Stage 1/2/3 ROP, normal) are "known."

**Rejection rule:** for a test embedding, compute distance to the nearest of the 6 known-class prototypes. If that distance exceeds a threshold, reject as "unknown"; otherwise assign the nearest known class.

**Inputs:** `eyeclip_embeddings.npz` (Phase 1) and the same split CSVs (with `categories` column) used in Phase 2/3. Fused representations reused from Phase 3 if available (`F_train`/`F_test` computed here for image-only; you can substitute Phase 3's fused arrays for the multimodal run).

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]
UNKNOWN_CLASSES = ["Stage 4 ROP", "Stage 5 ROP"]

EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")  # CHANGE to actual path -- must match Phase 1/2/3

data = np.load(EMB_PATH)
X_train_img, X_val_img, X_test_img = data["X_train"], data["X_val"], data["X_test"]

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

assert len(train_df) == len(X_train_img), "train.csv row count != X_train row count"
assert len(val_df)   == len(X_val_img),   "val.csv row count != X_val row count"
assert len(test_df)  == len(X_test_img),  "test.csv row count != X_test row count"

all_classes = sorted(train_df["categories"].unique())
for c in UNKNOWN_CLASSES:
    assert c in all_classes, f"'{c}' not found in categories -- check exact spelling: {all_classes}"
known_classes = [c for c in all_classes if c not in UNKNOWN_CLASSES]
print("known classes:", known_classes)
print("unknown classes:", UNKNOWN_CLASSES)

known classes: ['A-ROP', 'Laser-treated ROP', 'Stage 1 ROP', 'Stage 2 ROP', 'Stage 3 ROP', 'normal']
unknown classes: ['Stage 4 ROP', 'Stage 5 ROP']


In [3]:
# Build boolean masks: is_known / is_unknown per split, using string labels directly
# (avoids relying on Phase 1's integer label2idx, which we don't have access to here)
y_train_str = train_df["categories"].values
y_val_str   = val_df["categories"].values
y_test_str  = test_df["categories"].values

train_known_mask = np.isin(y_train_str, known_classes)
val_known_mask   = np.isin(y_val_str, known_classes)
test_known_mask  = np.isin(y_test_str, known_classes)

print(f"train: {train_known_mask.sum()} known / {(~train_known_mask).sum()} excluded-unknown")
print(f"val:   {val_known_mask.sum()} known / {(~val_known_mask).sum()} unknown")
print(f"test:  {test_known_mask.sum()} known / {(~test_known_mask).sum()} unknown")

# support pool for prototypes: only known-class TRAIN examples
X_train_known = X_train_img[train_known_mask]
y_train_known = y_train_str[train_known_mask]

train: 1351 known / 60 excluded-unknown
val:   277 known / 12 unknown
test:  303 known / 17 unknown


## Prototype construction + distance-to-nearest-known-prototype
Reuses the exact `build_prototypes` logic from Phase 2/3 (same seeds), but only over `known_classes`.

In [4]:
def build_prototypes(X, y, classes, k, rng):
    """Sample k examples per class (from known classes only) and average.
    Input:  X [N,512], y [N] string labels, classes list of known class names, k int
    Output: prototypes [C,512], in the same order as `classes`
    """
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)

def nearest_known_distance_and_pred(X, prototypes, classes, metric="cosine"):
    """For each row of X, return (min_distance_to_any_known_prototype, predicted_known_class).
    Distance is defined so LARGER = farther / more likely unknown, for both metrics.
    """
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        sim = X_n @ P_n.T                      # [N, C], higher = closer
        dist = 1.0 - sim                        # convert to distance: 0=identical, 2=opposite
    else:
        dist = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)  # [N, C]
    nearest_idx = dist.argmin(axis=1)
    min_dist = dist.min(axis=1)
    preds = np.array([classes[i] for i in nearest_idx])
    return min_dist, preds

## Threshold selection on VAL (not a fixed guess)
For each (shot, seed), sweep candidate thresholds over the val set's distance range and pick the one maximizing macro-F1 over {known classes + "unknown"} on val. That threshold is then applied, unchanged, to test.

In [5]:
def select_threshold(val_dist, val_is_known, n_candidates=50):
    """Sweep thresholds over val_dist's range, pick the one maximizing binary F1
    for the known-vs-unknown decision (reject if dist > threshold).
    Input:  val_dist [N] float, val_is_known [N] bool (True = actually a known-class image)
    Output: best threshold (float)
    """
    candidates = np.linspace(val_dist.min(), val_dist.max(), n_candidates)
    best_thresh, best_f1 = candidates[0], -1.0
    for t in candidates:
        pred_known = val_dist <= t   # predict "known" (accept) if distance within threshold
        f1 = f1_score(val_is_known, pred_known, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    return best_thresh

## Full open-set evaluation loop (image-only, across shots + seeds)
**Metrics:**
- **Known-F1**: macro-F1 over known-class predictions, computed only on test rows that are truly known AND were accepted (not rejected).
- **Unknown-F1**: F1 of the binary "is this rejected as unknown" decision (treats unknown-detection as its own binary task).
- **AUROC**: threshold-independent separability of known vs unknown using the raw distance score (computed once per run, doesn't depend on the chosen threshold).

In [6]:
def evaluate_open_set(X_train_known, y_train_known, X_val, val_is_known,
                       X_test, test_is_known, y_test_str, known_classes,
                       k, seed, metric="cosine"):
    rng = np.random.default_rng(seed)
    prototypes = build_prototypes(X_train_known, y_train_known, known_classes, k, rng)

    val_dist, _  = nearest_known_distance_and_pred(X_val,  prototypes, known_classes, metric)
    test_dist, test_preds = nearest_known_distance_and_pred(X_test, prototypes, known_classes, metric)

    threshold = select_threshold(val_dist, val_is_known)

    test_accepted = test_dist <= threshold   # True = model says "known", predicts a class

    # Unknown-F1: binary task, positive class = "is unknown" (rejected)
    unknown_f1 = f1_score(~test_is_known, ~test_accepted, zero_division=0)

    # Known-F1: macro-F1 over known-class labels, restricted to truly-known rows that were accepted
    eval_mask = test_is_known & test_accepted
    if eval_mask.sum() > 0:
        known_f1 = f1_score(y_test_str[eval_mask], test_preds[eval_mask],
                             labels=known_classes, average="macro", zero_division=0)
    else:
        known_f1 = 0.0  # nothing was accepted -- degenerate but must be reported, not hidden

    # AUROC: threshold-independent, uses raw distance as the "unknown score"
    try:
        auroc = roc_auc_score(~test_is_known, test_dist)  # higher dist should predict "unknown"
    except ValueError:
        auroc = float("nan")  # only one class present in y_true -- can't compute AUROC

    return {"threshold": threshold, "known_f1": known_f1, "unknown_f1": unknown_f1, "auroc": auroc}


open_set_results = []
for k in SHOTS:
    for seed in SEEDS:
        res = evaluate_open_set(
            X_train_known, y_train_known,
            X_val_img, val_known_mask,
            X_test_img, test_known_mask, y_test_str,
            known_classes, k, seed, metric="cosine")
        res.update({"shot": k, "seed": seed, "representation": "image_only"})
        open_set_results.append(res)

open_set_df = pd.DataFrame(open_set_results)
open_set_df

,threshold,known_f1,unknown_f1,auroc,shot,seed,representation
0,0.007258,0.158701,0.0,0.547758,1,0,image_only
1,0.003300,0.175282,0.0,0.467773,1,1,image_only
2,0.003810,0.157595,0.0,0.450980,1,2,image_only
3,0.004297,0.177879,0.0,0.461173,1,3,image_only
4,0.005812,0.127850,0.0,0.508057,1,4,image_only
5,0.004353,0.152279,0.0,0.447098,3,0,image_only
6,0.003573,0.164517,0.0,0.458455,3,1,image_only
7,0.007666,0.148979,0.0,0.496797,3,2,image_only
8,0.004702,0.116667,0.0,0.351291,3,3,image_only
9,0.006579,0.155124,0.0,0.466511,3,4,image_only


In [7]:
summary = open_set_df.groupby("shot").agg(
    known_f1_mean=("known_f1", "mean"), known_f1_std=("known_f1", "std"),
    unknown_f1_mean=("unknown_f1", "mean"), unknown_f1_std=("unknown_f1", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
    threshold_mean=("threshold", "mean"),
).reset_index()
print(summary.to_string(index=False))
summary.to_csv("phase4_openset_image_only.csv", index=False)

 shot  known_f1_mean  known_f1_std  unknown_f1_mean  unknown_f1_std  auroc_mean  auroc_std  threshold_mean
    1       0.159462      0.019955              0.0             0.0    0.487148   0.040215        0.004896
    3       0.147513      0.018189              0.0             0.0    0.444030   0.055018        0.005375
    5       0.150560      0.015205              0.0             0.0    0.502466   0.016532        0.007355


## Image+metadata (fused) open-set run
Requires Phase 3's fused representations. If you saved `F_train`/`F_test` (post-fusion-layer vectors) from the Phase 3 notebook, load or recompute them here, then rerun the same evaluation loop with fused features restricted to known-class train rows.

**Do not run this cell until you have fused train/val/test arrays available** -- Phase 3 only computed `F_train`/`F_test`, not `F_val`; you'll need to add an `F_val = get_fused_repr(Xi_val, Xm_val)` line in the Phase 3 notebook and re-save before this will work.

In [8]:
fused_data = np.load("fused_embeddings.npz")
F_train, F_val, F_test = fused_data["F_train"], fused_data["F_val"], fused_data["F_test"]

assert len(F_train) == len(X_train_img), "F_train row count doesn't match X_train -- wrong file/order?"
assert len(F_val)   == len(X_val_img),   "F_val row count doesn't match X_val"
assert len(F_test)  == len(X_test_img),  "F_test row count doesn't match X_test"

F_train_known = F_train[train_known_mask]

fused_results = []
for k in SHOTS:
    for seed in SEEDS:
        res = evaluate_open_set(F_train_known, y_train_known, F_val, val_known_mask,
                                 F_test, test_known_mask, y_test_str, known_classes, k, seed)
        res.update({"shot": k, "seed": seed, "representation": "image_plus_metadata"})
        fused_results.append(res)

fused_open_set_df = pd.DataFrame(fused_results)
fused_summary = fused_open_set_df.groupby("shot").agg(
    known_f1_mean=("known_f1", "mean"), known_f1_std=("known_f1", "std"),
    unknown_f1_mean=("unknown_f1", "mean"), unknown_f1_std=("unknown_f1", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
    threshold_mean=("threshold", "mean"),
).reset_index()
print(fused_summary.to_string(index=False))
fused_summary.to_csv("phase4_openset_fused.csv", index=False)

 shot  known_f1_mean  known_f1_std  unknown_f1_mean  unknown_f1_std  auroc_mean  auroc_std  threshold_mean
    1       0.199256      0.059094              0.0             0.0    0.457115   0.061863        0.365316
    3       0.220553      0.042040              0.0             0.0    0.542031   0.078853        0.266182
    5       0.219398      0.040471              0.0             0.0    0.587925   0.036657        0.260323


## Interpretation notes
- **Known-F1** tells you: among images the model accepted as known, how good is its classification (should stay in the same ballpark as Phase 2/3's few-shot numbers).
- **Unknown-F1** tells you: how well the rejection rule catches Stage 4/5 as "unknown" -- low values mean the model confidently (and wrongly) classifies unknowns as one of the 6 known classes.
- **AUROC** near 0.5 means known/unknown distances are not separable at all (rejection is not learnable with this method); near 1.0 means they're cleanly separable regardless of threshold choice.
- Given Phase 2 showed prototypes struggle even on known classes, expect open-set numbers to be modest -- report them as-is, don't tune the threshold search to chase a better number.
- Do **not** proceed to Phase 5 (text branch) until both image-only and image+metadata open-set tables are recorded.